In [ ]:
import os
from openai import OpenAI

os.environ["OPENAI_API_KEY"] = 

client = OpenAI()

response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "Give me a short definition of DNN."}
    ]
)

print(response.choices[0].message.content)

A Deep Neural Network (DNN) is a type of artificial neural network that consists of multiple layers of interconnected nodes, or neurons, which process data and learn patterns through the use of algorithms. DNNs are commonly used in various applications such as image and speech recognition, natural language processing, and more, leveraging their ability to model complex relationships in large datasets.


In [3]:
from openai import OpenAI

client = OpenAI()

# Simple evaluation dataset
eval_data = [
    {
        "input": "What is 2 + 2?",
        "expected_output": "4"
    },
    {
        "input": "What is the capital of France?",
        "expected_output": "Paris"
    }
]

def run_eval():
    for test_case in eval_data:
        response = client.responses.create(
            model="gpt-4.1-mini",
            input=test_case["input"]
        )

        model_output = response.output_text.strip()

        if model_output.lower() == test_case["expected_output"].lower():
            print(f"PASS: {test_case['input']}")
        else:
            print(f"FAIL: {test_case['input']}")
            print(f"Expected: {test_case['expected_output']}")
            print(f"Got: {model_output}")
            print()

run_eval()

FAIL: What is 2 + 2?
Expected: 4
Got: 2 + 2 equals 4.

FAIL: What is the capital of France?
Expected: Paris
Got: The capital of France is Paris.



In [4]:
from openai import OpenAI

client = OpenAI()

# Simple evaluation dataset
eval_data = [
    {
        "input": "What is 2 + 2? Reply with only the number.",
        "expected_output": "4"
    },
    {
        "input": "What is the capital of France?",
        "expected_output": "London"  # intentionally incorrect to force one FAIL
    }
]

def run_eval():
    for test_case in eval_data:
        response = client.responses.create(
            model="gpt-4.1-mini",
            input=test_case["input"]
        )

        model_output = response.output_text.strip()

        # More tolerant comparison
        if test_case["expected_output"].lower() in model_output.lower():
            print(f"PASS: {test_case['input']}")
        else:
            print(f"FAIL: {test_case['input']}")
            print(f"Expected: {test_case['expected_output']}")
            print(f"Got: {model_output}")
            print()

run_eval()

PASS: What is 2 + 2? Reply with only the number.
FAIL: What is the capital of France?
Expected: London
Got: The capital of France is Paris.



In [6]:
pip install openai pydantic

Note: you may need to restart the kernel to use updated packages.


You should consider upgrading via the 'c:\Users\Administrator\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [ ]:
import json
import uuid
from datetime import datetime
from typing import Dict

from pydantic import BaseModel, ValidationError
from openai import OpenAI

# -----------------------------
# CONFIG
# -----------------------------

os.environ["OPENAI_API_KEY"] = 
client = OpenAI()

PRIMARY_MODEL = "gpt-4.1"
JUDGE_MODEL = "gpt-4o-mini"

# -----------------------------
# SCHEMA FOR JUDGE OUTPUT
# -----------------------------

class JudgeScore(BaseModel):
    relevance: int
    faithfulness: int
    completeness: int
    overall_score: float


# -----------------------------
# PRIMARY LLM (System Under Test)
# -----------------------------

def generate_model_output(user_input: str, context: str) -> str:
    response = client.responses.create(
        model=PRIMARY_MODEL,
        input=[
            {
                "role": "system",
                "content": "You are a helpful enterprise assistant."
            },
            {
                "role": "user",
                "content": f"Context:\n{context}\n\nQuestion:\n{user_input}"
            }
        ]
    )

    return response.output_text


# -----------------------------
# BUILD JUDGE PROMPT
# -----------------------------

def build_judge_prompt(user_input: str, model_output: str, context: str) -> str:
    return f"""
You are an expert evaluator.

Evaluate the model output using the following metrics:
1. Relevance (1-5)
2. Faithfulness to context (1-5)
3. Completeness (1-5)

Return ONLY valid JSON in this format:
{{
  "relevance": int,
  "faithfulness": int,
  "completeness": int,
  "overall_score": float
}}

User Input:
{user_input}

Retrieved Context:
{context}

Model Output:
{model_output}
"""


# -----------------------------
# JUDGE LLM CALL
# -----------------------------

def run_judge(prompt: str) -> JudgeScore:
    response = client.responses.create(
        model=JUDGE_MODEL,
        temperature=0,  # deterministic scoring
        input=prompt
    )

    raw_output = response.output_text

    try:
        parsed = json.loads(raw_output)
        return JudgeScore(**parsed)

    except (json.JSONDecodeError, ValidationError) as e:
        print("Judge output invalid. Raw output:")
        print(raw_output)
        raise e


# -----------------------------
# STORE RESULTS (Simple Example)
# -----------------------------

def store_evaluation(result: Dict):
    with open("evaluation_log.jsonl", "a") as f:
        f.write(json.dumps(result) + "\n")


# -----------------------------
# MAIN PIPELINE
# -----------------------------

def evaluate_query(user_input: str, context: str):
    query_id = str(uuid.uuid4())
    timestamp = datetime.utcnow().isoformat()

    # 1️⃣ Generate model output
    model_output = generate_model_output(user_input, context)

    # 2️⃣ Build judge prompt
    judge_prompt = build_judge_prompt(user_input, model_output, context)

    # 3️⃣ Run judge
    judge_scores = run_judge(judge_prompt)

    # 4️⃣ Store evaluation result
    result_record = {
        "query_id": query_id,
        "timestamp": timestamp,
        "user_input": user_input,
        "model_output": model_output,
        "scores": judge_scores.model_dump(),
        "primary_model": PRIMARY_MODEL,
        "judge_model": JUDGE_MODEL
    }

    store_evaluation(result_record)

    return result_record


# -----------------------------
# EXAMPLE RUN
# -----------------------------

if __name__ == "__main__":
    context = "Company policy allows 15 days of paid leave per year."
    user_query = "How many vacation days do employees get?"

    result = evaluate_query(user_query, context)

    print("Evaluation Result:")
    print(json.dumps(result, indent=2))

Evaluation Result:
{
  "query_id": "0a9c42dc-fcd7-4d12-9470-f8b446dd4ca1",
  "timestamp": "2026-02-26T10:02:36.883702",
  "user_input": "How many vacation days do employees get?",
  "model_output": "Employees get 15 days of paid leave per year.",
  "scores": {
    "relevance": 5,
    "faithfulness": 5,
    "completeness": 5,
    "overall_score": 5.0
  },
  "primary_model": "gpt-4.1",
  "judge_model": "gpt-4o-mini"
}


### **AI E-Commerce Semantic Search with Orchestration + LLM Judge**

In [13]:
# Install required libraries
# sentence-transformers → embeddings + cross-encoder reranking
# faiss-cpu → vector similarity search
# openai → LLM + Judge model

!pip install sentence-transformers faiss-cpu openai

You should consider upgrading via the 'C:\Users\Administrator\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip' command.


In [16]:
# FAISS for vector similarity search
import faiss

# NumPy for vector manipulation
import numpy as np

# JSON for structured outputs
import json

# SentenceTransformer for embeddings
# CrossEncoder for reranking
from sentence_transformers import SentenceTransformer, CrossEncoder

# OpenAI client for LLM generation and judging
from openai import OpenAI

#from .autonotebook import tqdm as notebook_tqdm #newly added line by surendar while running the notebook 

In [17]:
# Simple product catalog (simulating a database)

products = [
    {
        "product_id": 1,
        "title": "Running Shoes",
        "description": "Comfortable lightweight running shoes for daily jogging.",
        "price": 4500
    },
    {
        "product_id": 2,
        "title": "Wireless Headphones",
        "description": "Noise cancelling headphones with 40-hour battery life.",
        "price": 3200
    },
    {
        "product_id": 3,
        "title": "Sports Watch",
        "description": "Waterproof sports watch with heart rate tracking.",
        "price": 2800
    }
]

In [20]:
# Simple word-based chunking function
# In production, you would use token-based chunking
# Here it’s simplified for demonstration

def chunk_text(text, chunk_size=50):
    words = text.split()
    return [" ".join(words[i:i+chunk_size]) for i in range(0, len(words), chunk_size)]

In [21]:
# Load embedding model
# all-MiniLM-L6-v2 produces 384-dimensional embeddings

embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 215.03it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [37]:
# Create empty list to store embeddings
embeddings = []

# Convert each product description into a vector
for product in products:
    vector = embedding_model.encode(product["description"])
    embeddings.append(vector)
    print(f"Generated embedding for product: {product['title']}")

# Convert to NumPy array and ensure float32 (required by FAISS)
embeddings = np.array(embeddings).astype("float32")
print(f"Embeddings: {embeddings}")

Generated embedding for product: Running Shoes
Generated embedding for product: Wireless Headphones
Generated embedding for product: Sports Watch
Embeddings: [[-0.03258745  0.03430683  0.00933359 ... -0.12329438  0.00288745
   0.02895475]
 [-0.01882266  0.08454173  0.02871769 ... -0.07849817 -0.00343516
   0.03804508]
 [-0.05817554  0.0650567   0.01284019 ... -0.04397825  0.00596645
   0.1077631 ]]


In [32]:
# Determine vector dimension (384)
dimension = embeddings.shape[1]
print(f"Embedding dimension: {dimension}")

# Use Inner Product index (after normalization → cosine similarity)
index = faiss.IndexFlatIP(dimension)

# Normalize vectors for cosine similarity
faiss.normalize_L2(embeddings)

# Add product embeddings to index
index.add(embeddings)

Embedding dimension: 384


In [25]:
def retrieve(query, top_k=3):
    # Convert query into embedding vector
    query_vector = embedding_model.encode(query).astype("float32")

    # Normalize query for cosine similarity
    faiss.normalize_L2(query_vector.reshape(1, -1))

    # Perform ANN search
    D, I = index.search(query_vector.reshape(1, -1), top_k)

    # Return corresponding products
    return [products[i] for i in I[0]]

In [26]:
# Load cross-encoder model for re-ranking
# This model scores query-product pairs more precisely

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

def rerank(query, candidates):
    # Create query-product pairs
    pairs = [(query, c["description"]) for c in candidates]

    # Predict relevance scores
    scores = reranker.predict(pairs)

    # Combine candidates with scores
    scored = list(zip(candidates, scores))

    # Sort descending by score
    scored.sort(key=lambda x: x[1], reverse=True)

    # Return sorted products
    return [item[0] for item in scored]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 160.06it/s, Materializing param=classifier.weight]                                    
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [28]:
# Initialize OpenAI client
client = OpenAI()

def generate_response(query, ranked_products):
    # Build context from retrieved products
    context = "\n".join([
        f"{p['title']} - {p['description']} - Price: {p['price']}"
        for p in ranked_products
    ])

    # Construct prompt
    prompt = f"""
You are an e-commerce assistant.
Only recommend from the provided products.

User Query:
{query}

Available Products:
{context}
"""

    # Call LLM
    response = client.responses.create(
        model="gpt-4o-mini",
        input=prompt
    )

    return response.output_text

In [29]:
def judge_response(query, ranked_products, answer):
    # Reconstruct product context
    context = "\n".join([
        f"{p['title']} - {p['description']} - Price: {p['price']}"
        for p in ranked_products
    ])

    # Judge prompt
    judge_prompt = f"""
Evaluate the answer based on:
1. Relevance (1-5)
2. Faithfulness to products (1-5)

Return JSON:
{{
  "relevance": int,
  "faithfulness": int
}}

Query:
{query}

Products:
{context}

Answer:
{answer}
"""

    # Deterministic judge call
    response = client.responses.create(
        model="gpt-4o-mini",
        temperature=0,
        input=judge_prompt
    )

    return response.output_text

In [30]:
class MiniEcommercePipeline:

    # Step 1: Retrieval
    def retrieve(self, query):
        print("Running Retrieval...")
        return retrieve(query)

    # Step 2: Re-ranking
    def rerank(self, query, candidates):
        print("Running Re-ranking...")
        return rerank(query, candidates)

    # Step 3: Generation
    def generate(self, query, ranked):
        print("Generating LLM Response...")
        return generate_response(query, ranked)

    # Step 4: Evaluation
    def evaluate(self, query, ranked, response):
        print("Running LLM Judge...")
        return judge_response(query, ranked, response)

    # Full pipeline orchestration
    def run(self, query):
        candidates = self.retrieve(query)
        ranked = self.rerank(query, candidates)
        response = self.generate(query, ranked)
        score = self.evaluate(query, ranked, response)
        return ranked, response, score

In [31]:
pipeline = MiniEcommercePipeline()

query = "comfortable running shoes under 5000"

ranked_products, llm_answer, evaluation = pipeline.run(query)

print("\nRanked Products:")
print(ranked_products)

print("\nLLM Answer:")
print(llm_answer)

print("\nEvaluation Score:")
print(evaluation)

Running Retrieval...
Running Re-ranking...
Generating LLM Response...
Running LLM Judge...

Ranked Products:
[{'product_id': 1, 'title': 'Running Shoes', 'description': 'Comfortable lightweight running shoes for daily jogging.', 'price': 4500}, {'product_id': 3, 'title': 'Sports Watch', 'description': 'Waterproof sports watch with heart rate tracking.', 'price': 2800}, {'product_id': 2, 'title': 'Wireless Headphones', 'description': 'Noise cancelling headphones with 40-hour battery life.', 'price': 3200}]

LLM Answer:
I recommend the **Comfortable lightweight running shoes for daily jogging** priced at **4500**. They fit your request for comfortable running shoes under 5000. Would you like more information about them?

Evaluation Score:
{
  "relevance": 5,
  "faithfulness": 5
}
